# MongoDB Assignment

### Task 1: Create and Insert Data
**Objective**: Learn how to create a new database and insert a single document into a collection.

**Description**: In this task, you will create a database called `school` and a collection named `students`. You will then insert a single student record into the collection. This teaches you the basic `use` command and `insertOne` method.

In [1]:
# If PyMongo is not installed, run this cell once:
# !pip install pymongo

from pymongo import MongoClient
from pprint import pprint

# Connect to local MongoDB server
client = MongoClient("mongodb://localhost:27017/")

# Create/use database: school
db = client["school"]

# Create/use collection: students
students = db["students"]

print("Connected successfully!")
print("Database:", db.name)
print("Collection:", students.name)


Connected successfully!
Database: school
Collection: students


In [2]:
# Start with clean data so the assignment gives the same result every time
students.delete_many({})

# Insert one student document
result = students.insert_one({
    "name": "Taha",
    "age": 20,
    "major": "Math"
})

print("Inserted document ID:", result.inserted_id)


Inserted document ID: 6a314fe356aaa13a45001638


In [3]:
# Show all documents after Task 1
for student in students.find({}, {"_id": 0}):
    pprint(student)


{'age': 20, 'major': 'Math', 'name': 'Taha'}


### Task 2: Insert Multiple Documents
**Objective**: Learn to insert multiple documents in a single command.

**Description**: You'll insert two additional student records into the `students` collection. This helps you understand how `insertMany` works to add multiple entries at once.

In [4]:
# Insert multiple student documents
result = students.insert_many([
    {
        "name": "Emma",
        "age": 22,
        "major": "Physics"
    },
    {
        "name": "Liam",
        "age": 21,
        "major": "Math"
    }
])

print("Inserted IDs:")
for inserted_id in result.inserted_ids:
    print(inserted_id)


Inserted IDs:
6a314fe356aaa13a45001639
6a314fe356aaa13a4500163a


In [5]:
# Show all students after insertMany
for student in students.find({}, {"_id": 0}):
    pprint(student)


{'age': 20, 'major': 'Math', 'name': 'Taha'}
{'age': 22, 'major': 'Physics', 'name': 'Emma'}
{'age': 21, 'major': 'Math', 'name': 'Liam'}


### Task 3: Querying Documents
**Objective**: Practice querying documents using specific conditions.

**Description**: You will write queries to find students who major in Math and those who are older than 20. This task introduces filtering documents with fields and comparison operators.

In [6]:
# Find students who major in Math
math_students = students.find({"major": "Math"}, {"_id": 0})

print("Students majoring in Math:")
for student in math_students:
    pprint(student)


Students majoring in Math:
{'age': 20, 'major': 'Math', 'name': 'Taha'}
{'age': 21, 'major': 'Math', 'name': 'Liam'}


In [7]:
# Find students older than 20
older_than_20 = students.find({"age": {"$gt": 20}}, {"_id": 0})

print("Students older than 20:")
for student in older_than_20:
    pprint(student)


Students older than 20:
{'age': 22, 'major': 'Physics', 'name': 'Emma'}
{'age': 21, 'major': 'Math', 'name': 'Liam'}


### Task 4: Update a Document
**Objective**: Learn how to update a document’s field.

**Description**: Update the `major` of the student named Noah to "Computer Science". This demonstrates the use of the `$set` operator in `updateOne`.

In [8]:
# Update Noah's major to Computer Science
result = students.update_one(
    {"name": "Noah"},
    {"$set": {"major": "Computer Science"}}
)

print("Matched documents:", result.matched_count)
print("Modified documents:", result.modified_count)


Matched documents: 0
Modified documents: 0


In [9]:
# Confirm Noah was updated
noah = students.find_one({"name": "Noah"}, {"_id": 0})
pprint(noah)


None


### Task 5: Remove a Field
**Objective**: Learn how to remove a field from a document.

**Description**: Remove the `age` field from Emma's document. This task introduces the `$unset` operator to delete specific fields from documents.

In [10]:
# Remove the age field from Emma's document
result = students.update_one(
    {"name": "Emma"},
    {"$unset": {"age": ""}}
)

print("Matched documents:", result.matched_count)
print("Modified documents:", result.modified_count)


Matched documents: 1
Modified documents: 1


In [11]:
# Confirm Emma's age field was removed
emma = students.find_one({"name": "Emma"}, {"_id": 0})
pprint(emma)


{'major': 'Physics', 'name': 'Emma'}


### Task 6: Use Operators for Advanced Queries
**Objective**: Use range conditions to query documents.

**Description**: Write a query to find all students whose ages are between 20 and 22 inclusive. This involves using `$gte` and `$lte` operators together.

In [12]:
# Find students whose age is between 20 and 22 inclusive
students_age_20_to_22 = students.find(
    {"age": {"$gte": 20, "$lte": 22}},
    {"_id": 0}
)

print("Students with age between 20 and 22 inclusive:")
for student in students_age_20_to_22:
    pprint(student)


Students with age between 20 and 22 inclusive:
{'age': 20, 'major': 'Math', 'name': 'Taha'}
{'age': 21, 'major': 'Math', 'name': 'Liam'}


### Task 7: Use Aggregation Pipeline
**Objective**: Group documents and perform counts.

**Description**: Count the number of students per major. This will demonstrate MongoDB's `aggregate` function and `$group` stage.

In [13]:
# Count number of students per major using aggregation pipeline
pipeline = [
    {
        "$group": {
            "_id": "$major",
            "numberOfStudents": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "numberOfStudents": -1,
            "_id": 1
        }
    }
]

result = students.aggregate(pipeline)

print("Number of students per major:")
for item in result:
    pprint(item)


Number of students per major:
{'_id': 'Math', 'numberOfStudents': 2}
{'_id': 'Physics', 'numberOfStudents': 1}


### Task 8: Field Comparison with $expr
**Objective**: Compare two fields in a document.

**Description**: Insert students with `marks` and `passingMarks`. Then find students whose `marks` are less than `passingMarks` using `$expr`.

In [14]:
# Insert students with marks and passingMarks
students.insert_many([
    {
        "name": "Sophia",
        "age": 19,
        "major": "Biology",
        "marks": 45,
        "passingMarks": 50
    },
    {
        "name": "Mason",
        "age": 23,
        "major": "Chemistry",
        "marks": 70,
        "passingMarks": 50
    }
])

print("Students with marks inserted successfully.")


Students with marks inserted successfully.


In [15]:
# Find students whose marks are less than passingMarks using $expr
failed_students = students.find(
    {"$expr": {"$lt": ["$marks", "$passingMarks"]}},
    {"_id": 0}
)

print("Students whose marks are less than passing marks:")
for student in failed_students:
    pprint(student)


Students whose marks are less than passing marks:
{'age': 19,
 'major': 'Biology',
 'marks': 45,
 'name': 'Sophia',
 'passingMarks': 50}


### Task 9: Use Array Queries
**Objective**: Query and update array fields.

**Description**: Insert a student with an array of courses. Then add a new course to the list and retrieve all students enrolled in a specific course. Learn how to use `$push` to update arrays and how to match documents based on array values.

In [16]:
# Insert a student with an array of courses
students.insert_one({
    "name": "Olivia",
    "age": 20,
    "major": "Computer Science",
    "courses": ["Database", "Python", "Data Structures"]
})

print("Student with courses inserted successfully.")


Student with courses inserted successfully.


In [17]:
# Add a new course to Olivia's courses array
students.update_one(
    {"name": "Olivia"},
    {"$push": {"courses": "MongoDB"}}
)

# Retrieve all students enrolled in MongoDB
mongodb_students = students.find(
    {"courses": "MongoDB"},
    {"_id": 0}
)

print("Students enrolled in MongoDB:")
for student in mongodb_students:
    pprint(student)


Students enrolled in MongoDB:
{'age': 20,
 'courses': ['Database', 'Python', 'Data Structures', 'MongoDB'],
 'major': 'Computer Science',
 'name': 'Olivia'}


## 📋 Important Execution Notes

* 🟢 **MongoDB Service Status**: Ensure that the local MongoDB service is actively running on your Windows machine before executing any code cells.
* 🔢 **Execution Order**: Run all cells sequentially from top to bottom to maintain the proper flow of data and state management.
* 🧹 **Data Idempotency**: The very first notebook execution cell clears and resets the `students` collection. This prevents data duplication and keeps tests consistent upon multiple re-runs.
* 🛠️ **Troubleshooting Dependencies**: If you encounter a `ModuleNotFoundError: No module named 'pymongo'` exception, uncomment and execute the `!pip install pymongo` line located within the first configuration cell.